In [ ]:
import sys,os
hsaDir = os.path.abspath("../../")
srcDir = os.path.join(hsaDir,"src")
sys.path.insert(0,srcDir)

In [ ]:
from heliostat_surface_analysis.loader.scaling_loader import ScalingLoader
from heliostat_surface_analysis.loader.default_loader import DefaultLoader
from heliostat_surface_analysis.preprocessor.basic_preprocessor import BasicPreprocessor
from heliostat_surface_analysis.preprocessor.no_preprocessor import NoPreprocessor
from heliostat_surface_analysis.mesher.delaunay_mesher import DelaunayMesher
from heliostat_surface_analysis.mesher.bspline_mesher import BSplineMesher
from heliostat_surface_analysis.analyser.fit_surface_analyser import FitSurfaceAnalyser
from heliostat_surface_analysis.analyser.fixed_focal_length_analyser import FixedFocalLengthAnalyser
from heliostat_surface_analysis.aggregator.aggregator import WeightedRMSAggregator
from heliostat_surface_analysis.analysis_manager import AnalysisManager
from heliostat_surface_analysis.plotting.plotting import plot_2d_surface,plot_3d_surface,set_plot_font_size

In [ ]:
import pandas as pd
import progressbar
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import random
import copy

In [ ]:
%matplotlib notebook

### Setup analysis components

In [ ]:
loader = DefaultLoader()
preprocessor = BasicPreprocessor()
mesher = DelaunayMesher({'max_aspect_ratio':10})
analyser = FitSurfaceAnalyser({"surface":"circular_paraboloid"})
aggregator = WeightedRMSAggregator()

In [ ]:
scansDir = os.path.join(hsaDir,"examples","command_line","Field1_stage4_2023","scans")
scanFiles = sorted(os.listdir(scansDir))
print(len(scanFiles))

### Analyse a random scan

In [ ]:
scanFile = random.choice(scanFiles)
scanPath = os.path.join(scansDir,scanFile)

In [ ]:
scan_name = os.path.splitext(os.path.basename(scanPath))[0]
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
mesh = mesher.create_mesh(x, y, z)
areas, normals, coeffs, ideal_normals, ideal_zs, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
sigma = aggregator.aggregate(slope_errors,areas)/np.sqrt(2)

f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.set_title("%s (%0.2f mrad)"%(scan_name,sigma*1e3))
plot_2d_surface(mesh,slope_errors,label="slope error (rad)",ax=ax,fontsize=16,cbxy=(4.0,0.29))
f,ax = plt.subplots()
f.set_size_inches(10,6)
ax.set_title("%s (%0.2f mrad)"%(scan_name,sigma*1e3))
ax.hist(slope_errors,bins=20,rwidth=0.85)
ax.grid()
ax.set_xlabel("slope error (rad)")
set_plot_font_size(ax,16,False)

## 3d surface plots

### measured surface coloured by slope error

In [ ]:
plot_3d_surface(mesh,slope_errors*1e3,"slope error (mrad)",fontsize=16)

### residual surface coloured by slope error

In [ ]:
dzs = (z-ideal_zs)*1e3
dz_mesh = mesher.create_mesh(x,y,dzs)

In [ ]:
fig = plt.figure()
fig.set_size_inches(10,10)
ax = fig.add_subplot(111,projection='3d')
plot_3d_surface(dz_mesh,values=slope_errors*1e3,label="slope error (mrad)",ax=ax,fontsize=16)
ax.set_zlabel("z (mm)")
plt.show()

### residual surface coloured by z residual

In [ ]:
fig = plt.figure()
fig.set_size_inches(10,10)
ax = fig.add_subplot(111,projection='3d')
plot_3d_surface(dz_mesh,values=np.mean(dz_mesh.z,axis=1),label="z residual (mm)",ax=ax,fontsize=16)
ax.set_zlabel("z (mm)")
plt.show()